In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from collections import Counter

def _parse_amenities(val):
    """Convertit la chaîne '{TV, Wifi, Kitchen}' en liste ['tv', 'wifi', 'kitchen']"""
    if pd.isna(val): return []
    val = str(val).strip().replace('{','[').replace('}',']').replace('"','"')
    try:
        return [i.strip().lower() for i in ast.literal_eval(val)]
    except:
        return [i.strip().lower() for i in val.strip('{}').replace('"','').split(',')]

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize']=(10,5)
pd.set_option("display.max_columns", None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
os.getcwd()
print(os.listdir("/content"))
os.chdir('/content/drive/MyDrive/Airbnb_2')
os.getcwd()

['.config', 'sample_data']


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MonDrive/Airbnb_2'

# Exploration

In [ ]:
airbnb = pd.read_csv("airbnb_train.csv")
airbnb_test=pd.read_csv("airbnb_test.csv")
print(f"Il y a {airbnb.shape[0]} logements pour {airbnb.shape[1]} colonnes")
airbnb.head()

FileNotFoundError: [Errno 2] No such file or directory: 'airbnb_train.csv'

In [ ]:
print("Colonnes du dataset :")
airbnb.columns


Type de données contenues dans le dataset

In [ ]:
airbnb.info()

On voit que la majorité des données stockées sont de type 'object'.

Nombre de valeurs nulles dans le dataset (par colonne)

In [ ]:
missing=airbnb.isnull().sum()
missing_pourcent=(missing/len(airbnb)*100).round(1) #calcul du pourcentage de valeurs manquantes
missing_dataframe=pd.DataFrame({'nb de valeurs manquantes' : missing, '%':missing_pourcent}) #nouveau DataFrame pour visualiser les valeurs manquantes
missing_dataframe=missing_dataframe[missing_dataframe['nb de valeurs manquantes']>0].sort_values('%', ascending=False)

#graphe pour rendre les observations plus visuelles
fig, axes=plt.subplots(figsize=(8,5))
graphe_barres=axes.barh(missing_dataframe.index, missing_dataframe['%'], color='salmon',edgecolor='white')
axes.set_xlabel('% valeurs manquantes')
axes.set_title('Valeurs manquantes par colonne', fontsize=14)
for barre, valeur in zip(graphe_barres, missing_dataframe['%']):
    axes.text(barre.get_width()+0.3, barre.get_y()+barre.get_height()/2,f'{valeur}%',va='center', fontsize=9) #ajout du texte à côté de chaque barre
plt.tight_layout() #ajustement des espacements pour la lisibilité
plt.show()

Le pourcentage de valeurs nulles étant assez important, on décide donc de drop les colonnes concernées.
On drop aussi 'description', car on sait d'ores et déjà qu'elle ne permettra pas d'améliorer les prédictions : aucune information utile sur l'appartement n'est donnée.

In [ ]:

airbnb.drop("description", axis=1, inplace=True)
airbnb.columns

Affichage de statistiques génériques (count, moyenne, écart-type, nombre d'élements uniques...)

In [ ]:
airbnb.describe(include="all")

## Analyse de la distribution des prix

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(airbnb['log_price'], bins=40, kde=True)
plt.title("Distribution du log_price")
plt.show()

On observe que la variable `log_price` suit une distribution relativement proche d’une loi normale.

L’utilisation du logarithme du prix permet de réduire l’impact des valeurs extrêmes présentes dans les prix Airbnb. Sans cette transformation, quelques logements très chers pourraient fortement perturber les modèles de régression.

Cette transformation rend également les relations entre variables plus linéaires, ce qui améliore généralement les performances des modèles de machine learning.

Ce graphique montre :

la répartition des prix,
la présence ou non d’outliers,
si la target est exploitable pour une régression

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(data=airbnb, x='city', y='log_price')
plt.xticks(rotation=45)
plt.title("Prix selon la ville")
plt.show()

## Influence de la ville sur le prix

Ce graphique montre que le prix des logements varie fortement selon la ville.

Certaines villes présentent :
- une médiane de prix plus élevée
- une plus grande dispersion des prix
- davantage de logements très chers

La variable `city` semble donc être une caractéristique importante pour prédire le prix d’un logement Airbnb.

Cela justifie sa conservation dans le modèle de prédiction.

Il permet :

d’identifier une variable importante
de justifier le feature engineering
de motiver l’encodage des villes

In [ ]:
numeric_cols = airbnb.select_dtypes(include=np.number)

plt.figure(figsize=(12,8))
sns.heatmap(numeric_cols.corr(), cmap='coolwarm')
plt.title("Matrice de corrélation")
plt.show()

## Corrélations entre variables numériques

La matrice de corrélation permet d’identifier les relations entre les différentes variables numériques.

On observe notamment que certaines caractéristiques comme :
- le nombre de personnes accueillies (`accommodates`)
- le nombre de chambres
- le nombre de lits

semblent positivement corrélées avec le prix du logement.

Cette analyse permet :
- d’identifier les variables les plus utiles
- de détecter d’éventuelles redondances entre variables
- de mieux comprendre les facteurs influençant les prix.

In [ ]:
sns.scatterplot(data=airbnb, x='accommodates', y='log_price')
plt.title("Capacité d'accueil vs prix")
plt.show()

## Relation entre capacité d’accueil et prix

On observe une tendance générale : plus un logement peut accueillir de personnes, plus son prix augmente.

La relation n’est cependant pas parfaitement linéaire, ce qui suggère que d’autres variables influencent également fortement le prix :
- la localisation
- les équipements
- le type de logement
- les notes des utilisateurs

Cette variable reste néanmoins pertinente pour la prédiction.

Fréquence d'apparition des valeurs dites 'top' dans les colonnes éligibles (sans compter les valeurs nulles)

In [ ]:
n = len(airbnb)

print(f"Pourcentage d'apparition de 'Real Bed' dans bed_type : {sum(airbnb[airbnb['bed_type'] == 'Real Bed'].count())/airbnb['bed_type'].count()}")
print(f"Pourcentage d'apparition de 'Entire home/apt' dans room_type : "
      f"{sum(airbnb[airbnb['room_type'] == 'Entire home/apt'].count())/airbnb['room_type'].count()}")
print(f"Pourcentage d'apparition de 'strict' dans bed_type : {sum(airbnb[airbnb['cancellation_policy'] == 'strict'].count())/airbnb['cancellation_policy'].count()}")
print(f"Pourcentage d'apparition de 'True' dans cleaning_fee : {sum(airbnb[airbnb['cleaning_fee'] == True].count())/airbnb['cleaning_fee'].count()}")
print(f"Pourcentage d'apparition de 't' dans host_has_profile_pic : "
      f"{sum(airbnb[airbnb['host_has_profile_pic'] == 't'].count())/airbnb['host_has_profile_pic'].count()}")
print(f"Pourcentage d'apparition de 't' dans host_identity_verified : "
      f"{sum(airbnb[airbnb['host_identity_verified'] == 't'].count())/airbnb['host_identity_verified'].count()}")
print(f"Pourcentage d'apparition de 'f' dans instant_bookable : {sum(airbnb[airbnb['instant_bookable'] == 'f'].count())/airbnb['instant_bookable'].count()}")
print(f"Pourcentage d'apparition de 'Williamsburg' dans neighbourhood : "
      f"{sum(airbnb[airbnb['neighbourhood'] == 'Williamsburg'].count())/airbnb['neighbourhood'].count()}")


On remarque que certains pourcentages sont très importants

Vérification des colonnes dont le contenu ne peut pas être traité par un algorithme

In [ ]:
for i in range(2,airbnb.shape[1]):
    print(f"{airbnb.columns[i]} : {airbnb.iloc[:,i].unique()}")

On peut transformer ces colonnes en index pour simplifier le traitement de l'information par l'algorithme : c'est la prochaine étape de ce notebook.

# Entraînement

### Remplacer le type de propriété par un indice, cela permet à l’algo de l’utiliser

In [ ]:
class CustomTransformation():

    rooms = airbnb["room_type"].unique()
    beds_type = airbnb["bed_type"].unique()
    cancellation = airbnb["cancellation_policy"].unique()
    verified = airbnb["host_identity_verified"].unique()
    online_booking = airbnb["instant_bookable"].unique()
    cities = airbnb["city"].unique()
    properties = airbnb["property_type"].unique()

    def __init__(self):
        """
        Class simple pour convertir les type de propriétés en des indices numériques, utilisable pour un algo de machine learning
        """

        self.fitted = False # Indique si fit_transform a été utilisé, pour éviter d’utiliser transform sans que fit ait été appelé
        self.property2index = dict()# Dictionnaire qui va convertir le nom en indice
        self.rooms2index = dict()
        self.beds2index = dict()
        self.cancel2index = dict()
        self.online2index = dict()
        self.cities2index = dict()

        self.tf2index = {'t':True, 'f':False}

        self.max_index_prop = 0 # Indique le dernier indice de la propriété.
        self.max_index_rooms = 0
        self.max_index_beds = 0
        self.max_index_cancel = 0
        self.max_index_online = 0
        self.max_index_cities = 0

    def fit_transform(self, dataset):
        self.fitted = True

        # Calcul des mappings uniquement
        self.property2index = {prop:i for (i, prop) in enumerate(self.properties)}
        self.max_index_prop = max(list(self.property2index.values()))

        self.rooms2index = {r:i for (i, r) in enumerate(self.rooms)}
        self.max_index_rooms = max(list(self.rooms2index.values()))

        self.beds2index = {b:i for (i, b) in enumerate(self.beds_type)}
        self.max_index_beds = max(list(self.beds2index.values()))

        self.cancel2index = {can:i for (i, can) in enumerate(self.cancellation)}
        self.max_index_cancel = max(list(self.cancel2index.values()))

        self.online2index = {on:i for (i, on) in enumerate(self.online_booking)}
        self.max_index_online = max(list(self.online2index.values()))

        self.cities2index = {cit:i for (i, cit) in enumerate(self.cities)}
        self.max_index_cities = max(list(self.cities2index.values()))

        # Calcul des top amenities (fit uniquement)
        dataset['amenities_list'] = dataset['amenities'].apply(_parse_amenities)
        all_items = [item for lst in dataset['amenities_list'] for item in lst]
        self.top_amenities = [item for item, _ in Counter(all_items).most_common(30)]
        dataset.drop('amenities_list', axis=1, inplace=True)

        # Tout le reste est fait dans transform
        return self.transform(dataset)

    def transform(self, dataset):
        # Transform les propriétés en indice
        dataset.loc[:, "property_type"] = dataset["property_type"].replace(self.property2index)
        dataset.loc[:, "room_type"] = dataset["room_type"].replace(self.rooms2index)
        dataset.loc[:, "bed_type"] = dataset["bed_type"].replace(self.beds2index)
        dataset.loc[:, "cancellation_policy"] = dataset["cancellation_policy"].replace(self.cancel2index)
        dataset.loc[:, "instant_bookable"] = dataset["instant_bookable"].replace(self.online2index)
        dataset.loc[:, "city"] = dataset["city"].replace(self.cities2index)
        dataset.loc[:, "property_type"] = dataset["property_type"].replace(self.property2index)

        dataset.loc[:, "host_has_profile_pic"] = dataset["host_has_profile_pic"].replace(self.tf2index)
        dataset.loc[:, "host_identity_verified"] = dataset["host_identity_verified"].replace(self.tf2index)
        dataset.loc[:, "instant_bookable"] = dataset["instant_bookable"].replace(self.tf2index)

        # Même traitement que fit_transform, mais sans recalculer les top amenities
        dataset['amenities_list'] = dataset['amenities'].apply(_parse_amenities)
        dataset['amenities_count'] = dataset['amenities_list'].apply(len)

        for amenity in self.top_amenities:
            col = 'amenity_' + amenity.replace(' ', '_').replace('/', '_')
            dataset[col] = dataset['amenities_list'].apply(lambda lst: int(amenity in lst))

        dataset.drop('amenities_list', axis=1, inplace=True)


         # Ligne un peu moche qui fait en sorte de remplacer les lignes qui ont des noms de logement qui n’était pas dans l’entrainement
        dataset.loc[dataset["property_type"].map(type).eq(str), "property_type"] = np.nan


        # remplace les valeurs null
        dataset[dataset.bathrooms.isna()] = 0
        dataset[dataset.accommodates.isna()] = 0
        dataset[dataset.property_type.isna()] = self.max_index_prop + 1
        dataset[dataset.room_type.isna()] = self.max_index_rooms + 1
        dataset[dataset.bed_type.isna()] = self.max_index_beds + 1
        dataset[dataset.cancellation_policy.isna()] = self.max_index_cancel + 1
        dataset[dataset.instant_bookable.isna()] = self.max_index_online + 1
        dataset[dataset.city.isna()] = self.max_index_cities + 1

        # Conversion des dates en nombre de jours
        ref = pd.Timestamp('2017-01-01')
        for col in ['host_since', 'first_review', 'last_review']:
            if col in dataset.columns:
                dataset[col] = pd.to_datetime(dataset[col], errors='coerce')
                dataset[col + '_days'] = (ref - dataset[col]).dt.days
                dataset[col + '_days'] = dataset[col + '_days'].fillna(dataset[col + '_days'].median())
                dataset.drop(col, axis=1, inplace=True)

        # host_response_rate : '90%' → 90.0
        if 'host_response_rate' in dataset.columns:
            dataset['host_response_rate'] = (
                dataset['host_response_rate']
                .astype(str)
                .str.replace('%', '', regex=False)
                .str.strip()
                .replace({'nan': np.nan, '': np.nan})
                .astype(float)
            )
            dataset['host_response_rate'] = dataset['host_response_rate'].fillna(
                dataset['host_response_rate'].median()
            )


        return dataset

In [ ]:
features_transformer = CustomTransformation()

airbnb.head()
airbnb_train = features_transformer.fit_transform(airbnb)
airbnb_train.head()

In [ ]:
class FeatureSelection():

    def __init__(self):
        """
        Class simple pour juste garder les colonnes qui nous intéresse
        N'a pas forcément l'air nécessaire, mais c'est pour être sur que j'applique bien le même process au train et au test
        """

    def fit_transform(self, dataset, y=None):
        return self.transform(dataset)

    def transform(self, dataset):
        cols = [
            # Taille du logement
            "accommodates", "bathrooms", "bedrooms", "beds",

            # Type de logement
            "property_type", "room_type", "bed_type",

            # Localisation
            "city", "latitude", "longitude",

            # Conditions
            "cancellation_policy", "cleaning_fee", "instant_bookable",

            # Hôte
            "host_has_profile_pic", "host_identity_verified",

            # Avis
            "number_of_reviews", "review_scores_rating",

            "host_since_days", "first_review_days", "last_review_days", "host_response_rate",
        ]

        # on garde que les colonnes qui existent réellement
        cols = [c for c in cols if c in dataset.columns]
        new_dataset = dataset[cols].copy()
        # Récupère automatiquement toutes les colonnes amenity_* créées
        amenity_cols = [c for c in dataset.columns if c.startswith("amenity_")]
        cols += amenity_cols
        cols += ["amenities_count"]

        for col in new_dataset.columns:
            new_dataset[col] = pd.to_numeric(new_dataset[col], errors='coerce').fillna(0)
        return new_dataset


feature_selector = FeatureSelection()

airbnb_train = feature_selector.transform(airbnb)
airbnb_train.head()

### Apprentissage

In [ ]:
from sklearn.model_selection import train_test_split

# Vous avez le droit d’utiliser les Pipeline et transform de sklearn :
# https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
features_transformer = CustomTransformation()
feature_selector = FeatureSelection()

airbnb = pd.read_csv("airbnb_train.csv")

airbnb_train = features_transformer.fit_transform(airbnb)
airbnb_train = feature_selector.transform(airbnb_train)

X = airbnb_train.copy()
y = airbnb["log_price"]

# train cross validation
X_train, X_test, y_train, y_test = train_test_split(X, y)

Le score R2 est un score de regression, il vaut 1 si la prédiction est parfaite, 0 si la valeur prédite est la moyenne de $y$. Et des scores négatifs si la prédiction est moins bonne que prédire la moyenne (donc vraiment mauvais)

In [ ]:
from sklearn.svm import LinearSVR
from sklearn.metrics import r2_score

model = LinearSVR()

model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)


print(f"Score en entrainenement : {r2_score(y_true=y_train, y_pred=y_pred_train)}")
print(f"Score en cross validation : {r2_score(y_true=y_test, y_pred=y_pred_test)}")


In [ ]:
errors = y_test - y_pred_test

sns.histplot(errors, bins=40)
plt.title("Distribution des erreurs")
plt.show()

## Analyse des erreurs du modèle

La distribution des erreurs permet d’évaluer la qualité globale du modèle.

Un bon modèle produit généralement :
- des erreurs centrées autour de zéro
- une distribution relativement symétrique
- peu de valeurs extrêmes

Les erreurs importantes peuvent correspondre à des logements atypiques ou à des informations insuffisantes dans les données.

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_test)
plt.xlabel("Valeurs réelles")
plt.ylabel("Prédictions")
plt.title("Prédictions vs réalité")
plt.show()

## Comparaison entre valeurs réelles et prédictions

Ce graphique compare les valeurs réelles aux prédictions du modèle.

Si les prédictions étaient parfaites, tous les points seraient alignés sur une diagonale.

On observe ici que le modèle capture correctement la tendance générale mais présente encore certaines erreurs sur des logements particuliers.

Ce graphique permet de visualiser :
- la précision globale du modèle
- les éventuels biais
- la dispersion des prédictions.

## Prédiction sur le fichier de test

In [ ]:
airbnb_test = pd.read_csv("airbnb_test.csv")

# J’applique le même traitement que mon fichier entraînement
final_X_test = features_transformer.transform(airbnb_test)
final_X_test = feature_selector.transform(final_X_test)

final_X_test.tail()

In [ ]:
y_final_prediction = model.predict(final_X_test)
print(y_final_prediction)

## Sauvegarde dans le fichier de prédiction

In [ ]:
prediction_example = pd.read_csv("prediction_example.csv")
prediction_example["log_price"] = y_final_prediction

prediction_example.to_csv("MaPredictionFinale.csv", index=False) # index=False pour éviter d’ajouter l’index interne à pandas
# Voilà !

## Test de votre fichier

In [ ]:
def estConforme(monFichier_csv):
    votre_prediction = pd.read_csv(monFichier_csv)

    fichier_exemple = pd.read_csv("prediction_example.csv")

    assert votre_prediction.columns[1] == fichier_exemple.columns[1], f"Attention, votre colonne de prédiction doit s'appeler {fichier_exemple.columns[1]}, elle s'appelle '{votre_prediction.columns[1]}'"
    assert len(votre_prediction) == len(fichier_exemple), f"Attention, vous devriez avoir {len(fichier_exemple)} prédiction dans votre fichier, il en contient '{len(votre_prediction)}'"

    assert np.all(votre_prediction.iloc[:,0] == fichier_exemple.iloc[:, 0])

    print("Fichier conforme!")

estConforme("MaPredictionFinale.csv")

# Ce que je vais faire de mon côté

In [ ]:
# Vous n’avez pas accès à ce fichier, c’est normal, ce sont les vrais prédictions
# ===============================================================================
# true_test = pd.read_csv("../true_prediction.csv")
# ==========================================================

# votre_prediction = pd.read_csv("MaPredictionFinale.csv")["logpred"]
# print(r2_score(y_pred=votre_prediction, y_true=true_test["log_price"]))

# votre_prediction = pd.read_csv("prediction_example.csv")["logpred"] # Devrait avoir ~0
# print(r2_score(y_pred=votre_prediction, y_true=true_test["log_price"]))